In [0]:
from pyspark.sql.functions import col, count, avg, concat_ws, lit, round as spark_round

# Get the existing Spark session for serverless compute
try:
    from pyspark.sql import SparkSession
    spark = SparkSession.getActiveSession()
    if spark is None:
        raise RuntimeError("No active Spark session - this file must run in a Databricks environment")
except Exception as e:
    print(f"Error getting Spark session: {e}")
    raise

spark.sql("USE CATALOG arthasetu")

# ── FIX 1: Aggregate rural profiles by occupation ──
# Instead of 40k raw rows, create occupation summaries
# Much more useful for RAG — "typical tailoring borrower in UP"
print("[Fix 1] Aggregating rural profiles by occupation...")

spark.sql("""
    SELECT
        primary_business                               AS occupation,
        city,
        COUNT(*)                                       AS borrower_count,
        ROUND(AVG(annual_income), 0)                   AS avg_annual_income,
        ROUND(AVG(monthly_expenses), 0)                AS avg_monthly_expenses,
        ROUND(AVG(loan_amount), 0)                     AS avg_loan_amount,
        ROUND(AVG(loan_tenure), 0)                     AS avg_loan_tenure,
        ROUND(AVG(young_dependents + old_dependents),1) AS avg_dependents,
        CONCAT(
            'Occupation: ', primary_business, ' | ',
            'City: ', city, ' | ',
            'Typical borrowers: ', COUNT(*), ' | ',
            'Average annual income: Rs ', ROUND(AVG(annual_income),0), ' | ',
            'Average monthly expenses: Rs ', ROUND(AVG(monthly_expenses),0), ' | ',
            'Average loan amount needed: Rs ', ROUND(AVG(loan_amount),0), ' | ',
            'Average loan tenure: ', ROUND(AVG(loan_tenure),0), ' months | ',
            'Average dependents: ', ROUND(AVG(young_dependents + old_dependents),1)
        )                                              AS rag_text
    FROM arthasetu.bronze.rural_loan_raw
    GROUP BY primary_business, city
    HAVING COUNT(*) >= 5
    ORDER BY borrower_count DESC
""").write.format("delta").mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable("arthasetu.silver.rural_profiles_agg")

cnt = spark.sql("SELECT count(*) as c FROM arthasetu.silver.rural_profiles_agg").collect()[0]['c']
print(f"  ✅ arthasetu.silver.rural_profiles_agg → {cnt} rows (was 40,000)")
print("  Sample occupations:")
spark.sql("""
    SELECT occupation, city, borrower_count, avg_annual_income, avg_loan_amount
    FROM arthasetu.silver.rural_profiles_agg
    ORDER BY borrower_count DESC
    LIMIT 10
""").show(truncate=False)

# ── FIX 2: Make BhashaBench RAG text readable ───
# Combine question + correct answer option into plain text
print("\n[Fix 2] Processing BhashaBench Q&A for RAG...")

spark.sql("""
    SELECT
        id,
        question,
        correct_answer,
        topic,
        subject_domain,
        question_level,
        lang,
        CASE correct_answer
            WHEN 'A' THEN CONCAT('Q: ', question, ' | Answer: ', option_a)
            WHEN 'B' THEN CONCAT('Q: ', question, ' | Answer: ', option_b)
            WHEN 'C' THEN CONCAT('Q: ', question, ' | Answer: ', option_c)
            WHEN 'D' THEN CONCAT('Q: ', question, ' | Answer: ', option_d)
            ELSE          CONCAT('Q: ', question, ' | Answer: ', correct_answer)
        END                                            AS rag_text
    FROM arthasetu.bronze.bhashbench_finance_raw
    WHERE question IS NOT NULL
      AND topic IN (
            'Commerce','Banking','Finance','Economics',
            'Business Studies','Accounting','Insurance',
            'Taxation','Investment','Microfinance'
      )
""").write.format("delta").mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable("arthasetu.silver.finance_qa")

cnt = spark.sql("SELECT count(*) as c FROM arthasetu.silver.finance_qa").collect()[0]['c']
print(f"  ✅ arthasetu.silver.finance_qa → {cnt} rows")
spark.sql("""
    SELECT rag_text FROM arthasetu.silver.finance_qa LIMIT 3
""").show(truncate=80, vertical=True)

# ── Final Silver summary ─────────────────────────
print("\n" + "="*55)
print("SILVER — FINAL STATE")
print("="*55)
tables = {
    "loan_schemes"       : "Loan schemes with eligibility",
    "finance_qa"         : "Finance Q&A readable text",
    "rural_profiles_agg" : "Rural borrower occupation summaries",
    "state_context"      : "State credit penetration"
}
for t, desc in tables.items():
    try:
        cnt = spark.sql(f"SELECT count(*) as c FROM arthasetu.silver.{t}").collect()[0]['c']
        print(f"  ✅ {t:30s} {cnt:>6} rows — {desc}")
    except:
        print(f"  ❌ {t:30s}      0 rows — {desc} (not yet created)")
print("="*55)
print("✅ Silver processing complete")